<a href="https://colab.research.google.com/github/LGDCAR/apresentacao-historytelling/blob/main/postechchallenge_fase2_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 POSTECH  CHALLENGE   - FASE 2                                                                                          Machine Learning Applied to Business Machine

##Introdução

 Neste notebook, realizamos uma Análise Exploratória de Dados (AED) no conjunto de dados de Qualidade do Vinho. O objetivo principal desta análise é obter uma compreensão abrangente da estrutura do conjunto de dados, identificar padrões e distribuições entre as variáveis ​​e explorar como diversas propriedades químicas estão associadas às pontuações de qualidade do vinho.

Links importantes:
https://www.kaggle.com/datasets/yasserh/wine-quality-dataset

https://pandas.pydata.org/


In [ ]:
# Importando as bibliotecas e Carregando o dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, precision_score, recall_score


df = pd.read_csv('WineQT.csv')

In [ ]:
# Nesta seção, a estrutura do conjunto de dados é examinada para obter uma compreensão inicial dos dados disponíveis.
# Os nomes das colunas, os tipos de dados e a presença de valores ausentes são inspecionados para avaliar a qualidade e
# a consistência dos dados.

In [ ]:
df.head()

In [ ]:
# O código df.head() é usado para exibir as primeiras 5 linhas do seu DataFrame chamado df.
# Isso é útil para ter uma visão rápida da estrutura dos dados, das colunas e dos tipos de valores que ele contém,
# permitindo uma inspeção inicial do conjunto de dados.

In [ ]:
df.tail()

In [ ]:
# O código df.tail() é usado para exibir as últimas 5 linhas do seu DataFrame chamado df. Isso é útil para ter uma visão rápida do final dos dados,
# confirmando se o carregamento foi completo ou verificando a estrutura dos dados nas últimas entradas.

In [ ]:
df.describe()

In [ ]:
# O código df.describe() fornece um resumo estatístico das colunas numéricas do DataFrame df. Ele calcula estatísticas como
# contagem de valores não nulos (count), média (mean), desvio padrão (std),
# valor mínimo (min), valor máximo (max) e os quartis (25%, 50% - mediana, e 75%).
# Isso ajuda a obter uma compreensão rápida da distribuição e das características principais de cada variável numérica no seu conjunto de dados.

In [ ]:
df.info()

In [ ]:
# O código df.info() fornece um resumo conciso do DataFrame df. Ele exibe informações como o número total de entradas, o número de
# colunas, os nomes das colunas, a contagem de valores não nulos para cada coluna, o tipo de dado de cada coluna (dtype) e o uso de
# memória. Isso é extremamente útil para ter uma visão geral da estrutura do seu dataset e para verificar rapidamente se há valores
# ausentes (missing values).

In [ ]:
df.shape

O código df.shape retorna uma dupla que representa as dimensões do seu DataFrame df. O primeiro número na dupla indica o número de linhas (entradas/observações) e o segundo número indica o número de colunas (variáveis/características) no seu conjunto de dados. É uma maneira rápida de verificar o tamanho do seu DataFrame.

In [ ]:
df.isnull().sum()

In [ ]:
# O código df.isnull().sum() é usado para identificar e contar o número total de valores ausentes (nulos) em cada coluna do seu DataFrame
# df. Primeiro, df.isnull() cria um DataFrame booleano onde True indica um valor nulo e False indica um valor presente. Em seguida, .sum()
# soma esses valores booleanos (onde True é contado como 1 e False como 0) para cada coluna, resultando na contagem de valores
# ausentes por coluna. Isso é essencial para entender a completude dos seus dados.

In [ ]:
df[df.duplicated()]

O código df[df.duplicated()] é usado para identificar e exibir todas as linhas duplicadas presentes no DataFrame df. A função df.duplicated() retorna uma série booleana, onde True indica que a linha é uma duplicata (ou seja, ela já apareceu anteriormente no DataFrame) e False indica que a linha é única. Ao usar essa série booleana para indexar o DataFrame (df[...]), é feito um filtr  no DataFrame para mostrar apenas as linhas que foram marcadas como duplicadas.

Nesse caso, a saída foi Empty DataFrame, o que significa que não foram encontradas linhas duplicadas no  DataFrame df. Todos os registros são únicos.

In [ ]:
print("\n" + "="*40 + "\nETAPA 1: COMPREENSÃO DO PROBLEMA\n" + "="*40)


In [ ]:
# 1.1 Remover coluna irrelevante ('Id') se ela existir no arquivo do Kaggle
if 'Id' in df.columns:
    df = df.drop(columns=['Id'])
    print("-> Coluna 'Id' removida, pois não ajuda na modelagem.")

In [ ]:
# 1.2 Definindo claramente a variável alvo original
print(# Conteúdo original da coluna Alvo
f"Valores únicos originais da variável alvo (quality): {sorted(df['quality'].unique())}")

In [ ]:
# 1.3 Transformação da variável de qualidade em classificação binária
# Regra: Notas menor ou igual a 5 vira 0 (Baixa Qualidade). Notas 6 ou mais vira 1 (Alta Qualidade).
df['quality_binaria'] = df['quality'].apply(lambda x: 1 if x >= 7 else 0)

print("\n-> Transformação binária concluída com sucesso!")
print(df[['quality', 'quality_binaria']].head(10))


In [ ]:
# ==============================================================================
# ETAPA 2: ANÁLISE EXPLORATÓRIA DE DADOS (EDA)
# ==============================================================================
print("\n" + "="*40 + "\nETAPA 2: ANÁLISE EXPLORATÓRIA DE DADOS (EDA)\n" + "="*40)

In [ ]:
# 2.1 Investigar a distribuição das variáveis numéricas principais
print("-> Gerando gráficos de distribuição das características químicas...")
features_para_plotar = ['fixed acidity', 'volatile acidity', 'residual sugar', 'chlorides', 'alcohol']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(features_para_plotar):
    sns.histplot(df[col], kde=True, ax=axes[i], color='purple')
    axes[i].set_title(f'Distribuição de: {col}')

In [ ]:
# 2.2 Identificar correlações entre as variáveis (Matriz de Correlação)
print("\n-> Calculando a matriz de correlação...")
# Removemos a alvo categórica para a matriz pura
matriz_corr = df.drop(columns=['quality_binaria']).corr()


In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(matriz_corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Matriz de Correlação das Características do Vinho")
plt.show()


In [ ]:
print("""
Justificativas teóricas sugeridas para análise:
- Alta correlação positiva entre 'alcohol' e 'quality': O teor alcoólico costuma agradar mais aos avaliadores.
- Alta correlação negativa entre 'volatile acidity' (Ácido acético) e 'quality': Altos níveis dão gosto de vinagre ao vinho.
""")

In [ ]:
# 2.3 Detectar possíveis outliers (Valores discrepantes)
print("\n-> Gerando Boxplots para detecção de Outliers...")
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(features_para_plotar):
    sns.boxplot(x='quality_binaria', y=col, data=df, ax=axes[i], palette='Set2')
    axes[i].set_title(f'Outliers em: {col}')

axes[5].axis('off') # Desliga o último gráfico vazio
plt.tight_layout()
plt.show()

In [ ]:
# 2.4 Analisar o balanceamento das classes criadas
print("\n-> Verificando o balanceamento da Nova Variável Alvo Binária:")
contagem_classes = df['quality_binaria'].value_counts()
porcentagem_classes = df['quality_binaria'].value_counts(normalize=True) * 100

for classe, qtd in contagem_classes.items():
    lbl = "Alta Qualidade (1)" if classe == 1 else "Baixa Qualidade (0)"
    print(f"   Classe {lbl}: {qtd} amostras ({porcentagem_classes[classe]:.2f}%)")

plt.figure(figsize=(6, 4))
sns.countplot(x='quality_binaria', data=df, palette='viridis')
plt.title("Verificação de Balanceamento (Target Binário)")
plt.xticks([0, 1], ['Baixa Qualidade (0)', 'Alta Qualidade (1)'])
plt.show()

In [ ]:
# ==============================================================================
# ETAPA 3: PRÉ-PROCESSAMENTO DE DADOS
# ==============================================================================
print("\n" + "="*40 + "\nETAPA 3: PRÉ-PROCESSAMENTO DE DADOS\n" + "="*40)

# 3.1 Tratamento de dados faltantes (se houver)
valores_nulos = df.isnull().sum().sum()
print(f"-> Total de valores faltantes encontrados no dataset: {valores_nulos}")

if valores_nulos > 0:
    # Se existirem dados nulos, preenchemos com a mediana de cada coluna
    df = df.fillna(df.median())
    print("   Valores faltantes preenchidos com a mediana de cada coluna.")
else:
    print("   Nenhum tratamento de dados faltantes foi necessário.")

# Separando as variáveis explicativas (X) da variável alvo (y)
# Removemos 'quality' (alvo antigo) e 'quality_binaria' (alvo novo) de X
X = df.drop(columns=['quality', 'quality_binaria'])
y = df['quality_binaria']

# Divisão em Treino (80%) e Teste (20%) para garantir uma validação justa
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"-> Dados divididos: Treino = {X_train.shape[0]} amostras | Teste = {X_test.shape[0]} amostras")

# 3.2 Normalização ou Padronização de variáveis numéricas
# Usaremos o StandardScaler para deixar todas as variáveis na mesma escala (média 0 e desvio padrão 1)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("-> Padronização das variáveis numéricas concluída (StandardScaler).")

# 3.3 Criação de novas features (Feature Engineering)
# Exemplo relevante: Criar uma razão entre acidez volátil e acidez fixa
# Vinhos com muita acidez volátil em relação à fixa tendem a estragar o sabor
X_train_enhanced = X_train.copy()
X_test_enhanced = X_test.copy()

X_train_enhanced['acid_ratio'] = X_train_enhanced['volatile acidity'] / (X_train_enhanced['fixed acidity'] + 1e-5)
X_test_enhanced['acid_ratio'] = X_test_enhanced['volatile acidity'] / (X_test_enhanced['fixed acidity'] + 1e-5)

# Re-escalando os dados agora com a nova feature incluída
X_train_ready = scaler.fit_transform(X_train_enhanced)
X_test_ready = scaler.transform(X_test_enhanced)
print("-> Engenharia de Features concluída: Nova variável 'acid_ratio' adicionada.")

In [ ]:
# ETAPA 4: DESENVOLVIMENTO DE MODELOS
# ==============================================================================
print("\n" + "="*40 + "\nETAPA 4: DESENVOLVIMENTO DE MODELOS\n" + "="*40)

# Importando dois algoritmos diferentes de classificação para comparar
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# 4.1 Treinando o Modelo 1: Regressão Logística (Modelo Linear/Mais simples)
print("-> Treinando o Modelo 1: Regressão Logística...")
modelo_lr = LogisticRegression(random_state=42, max_iter=1000)
modelo_lr.fit(X_train_ready, y_train)

# 4.2 Treinando o Modelo 2: Floresta Aleatória (Modelo Baseado em Árvores/Mais complexo)
print("-> Treinando o Modelo 2: Floresta Aleatória (Random Forest)...")
modelo_rf = RandomForestClassifier(random_state=42, n_estimators=100)
modelo_rf.fit(X_train_ready, y_train)

print("\n-> Ambos os modelos foram treinados com sucesso e estão prontos para avaliação!")

In [ ]:
# ==============================================================================
# ETAPA 5: AVALIAÇÃO DOS MODELOS
# ==============================================================================
print("\n" + "="*40 + "\nETAPA 5: AVALIAÇÃO DOS MODELOS\n" + "="*40)

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

# 5.1 Realizando as previsões na base de teste
y_pred_lr = modelo_lr.predict(X_test_ready)
y_pred_rf = modelo_rf.predict(X_test_ready)

y_prob_lr = modelo_lr.predict_proba(X_test_ready)[:, 1]
y_prob_rf = modelo_rf.predict_proba(X_test_ready)[:, 1]

# 5.2 Exibindo o relatório de métricas para a Regressão Logística
print("\n[MÉTRICAS] - Modelo 1: Regressão Logística")
print(classification_report(y_test, y_pred_lr, target_names=['Baixa Qualidade (0)', 'Alta Qualidade (1)']))

# 5.3 Exibindo o relatório de métricas para a Random Forest
print("\n[MÉTRICAS] - Modelo 2: Random Forest")
print(classification_report(y_test, y_pred_rf, target_names=['Baixa Qualidade (0)', 'Alta Qualidade (1)']))

# 5.4 Plotando as Matrizes de Confusão e Curva ROC lado a lado
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Matriz LR
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lr, display_labels=['Baixa (0)', 'Alta (1)'], ax=axes[0], cmap='Blues', colorbar=False
)
axes[0].set_title("Matriz de Confusão - Regressão Logística")

# Matriz RF
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_rf, display_labels=['Baixa (0)', 'Alta (1)'], ax=axes[1], cmap='Purples', colorbar=False
)
axes[1].set_title("Matriz de Confusão - Random Forest")

# Curvas ROC
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

axes[2].plot(fpr_lr, tpr_lr, label=f'Regr. Logística (AUC = {auc(fpr_lr, tpr_lr):.2f})', color='blue')
axes[2].plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {auc(fpr_rf, tpr_rf):.2f})', color='purple')
axes[2].plot([0, 1], [0, 1], 'k--', label='Chute Aleatório')
axes[2].set_xlabel('Taxa de Falsos Positivos')
axes[2].set_ylabel('Taxa de Verdadeiros Positivos')
axes[2].set_title('Comparação de Curvas ROC')
axes[2].legend(loc='lower right')

plt.tight_layout()
plt.show()



In [ ]:
# ==============================================================================
# ETAPA 6: INTERPRETAÇÃO DOS RESULTADOS
# ==============================================================================
print("\n" + "="*40 + "\nETAPA 6: INTERPRETAÇÃO DOS RESULTADOS\n" + "="*40)

# Pegando os nomes das colunas atualizados (incluindo a nova feature criada)
nomes_colunas = X_train_enhanced.columns

# 6.1 Extraindo a importância das features do melhor modelo (normalmente a Random Forest)
importancias = modelo_rf.feature_importances_

# Criando um DataFrame para organizar e ordenar o ranking
df_importancia = pd.DataFrame({
    'Característica Química': nomes_colunas,
    'Importância': importancias
}).sort_values(by='Importância', ascending=False)

print("\n-> Ranking de influência das variáveis na qualidade do vinho (Random Forest):")
print(df_importancia.to_string(index=False))

# 6.2 Plotando o gráfico de importância de variáveis
plt.figure(figsize=(10, 6))
sns.barplot(x='Importância', y='Característica Química', data=df_importancia, palette='viridis')
plt.title('Quais componentes químicos mandam na qualidade do vinho?')
plt.xlabel('Nível de Importância Técnico')
plt.ylabel('Componente Químico')
plt.show()